# FairFace Inference Demo

## Imports

In [ ]:
import torch
import matplotlib.pyplot as plt

import os
import csv

from PIL import Image

from fairface_vit import FairFaceViT

from model.siglip import model as siglip_model, processor as siglip_processor
from model.clip   import model as clip_model,   processor as clip_processor
from model.dinov2 import model as dinov2_model, processor as dinov2_processor

## Class Labels

In [2]:
gender_class = ["Male", "Female"]

age_class = ["0-2", "3-9", "10-19", "20-29", "30-39", "40-49",
              "50-59", "60-69", "more than 70"]

race_class = ["East Asian", "Indian", "Black", "White", "Middle Eastern",
               "Latino_Hispanic", "Southeast Asian"]

## Utility Functions

### Checkpoint Loading

In [3]:
def load_checkpoint(model, model_name, run_version, device):
    """
    Load the trained classification heads from a saved checkpoint.
    """
    checkpoint = torch.load(
        f"../checkpoints/{model_name}/{run_version}/{model_name}-best-heads.pt",
        map_location=device
    )

    model.gender.load_state_dict(
        checkpoint["gender_head"]
    )

    model.age.load_state_dict(
        checkpoint["age_head"]
    )

    model.race.load_state_dict(
        checkpoint["race_head"]
    )

    model.to(device)
    model.eval()

    return model

### Core Inference

The model outputs logits, which are converted into probabilities:

- Sigmoid is used for binary gender classification.
- Softmax is used for multi-class age and race classification.

The class with the highest probability is selected as the final prediction.

In [4]:
def get_logits(model, processor, image, device):
    """
    Run inference on a single image and extract raw logits.

    Returns:
        Dictionary containing raw logits for each task.
    """
    # print(processor.to_dict())

    inputs = processor(
        images=image,
        return_tensors="pt"
    )

    pixel_values = inputs["pixel_values"].to(device)

    with torch.no_grad():
        outputs = model(pixel_values)

    # LOGITS
    return {
        'gender': outputs["gender"],
        'age': outputs["age"],
        'race': outputs["race"]
    }

In [5]:
def get_prediction_results(logits):
    """
    Convert model logits into human-readable predictions.

    Applies:
    - Sigmoid for binary gender classification.
    - Softmax for age and race multi-class classification.

    The class with the highest probability is selected as prediction.

    Args:
        logits: Dictionary containing model outputs.

    Returns:
        Dictionary containing predicted classes and confidence scores.
    """

    # --- Gender (binary, sigmoid) ---
    gender_prob = torch.sigmoid(logits['gender']).item()
    gender_idx = int(gender_prob > 0.5)
    gender_probs = [1 - gender_prob, gender_prob]

    # --- Age (multi-class, softmax) ---
    age_tensor = torch.softmax(logits['age'], dim=1)[0].cpu()
    age_idx = age_tensor.argmax().item()
    age_probs = age_tensor.tolist()

    # --- Race (multi-class, softmax) ---
    race_tensor = torch.softmax(logits['race'], dim=1)[0].cpu()
    race_idx = race_tensor.argmax().item()
    race_probs = race_tensor.tolist()

    return {
        'gender':      gender_class[gender_idx],
        'gender_prob': gender_probs[gender_idx],

        'age':         age_class[age_idx],
        'age_prob':    age_probs[age_idx],

        'race':        race_class[race_idx],
        'race_prob':   race_probs[race_idx],
    }


def print_ranked(category, class_names, probabilities):
    """
    Display class probabilities in descending order.

    Used for visualizing single image predictions.

    Args:
        category: Prediction category name.
        class_names: List of class labels.
        probabilities: Probability for each class.

    Returns:
        The class with the highest probability.
    """
    print(f"\n{category}:")

    pairs = []
    for i in range(len(class_names)):
        name = class_names[i]
        prob = probabilities[i]
        pairs.append((name, prob))

    pairs.sort(key=lambda pair: pair[1], reverse=True)


    for pair in pairs:
        name = pair[0]
        prob = pair[1]
        percentage = prob * 100
        print(f"  {name:<20}: {percentage:.2f}%")

    top_pair = pairs[0]
    top_name = top_pair[0]

    print(f"  -> Prediction: {top_name}")

    return top_name


### Visualization Utilities

In [6]:
def display_prediction(logits,name=None):
    """
    Display prediction results for a single image.

    Shows probability ranking for:
    - Gender
    - Age
    - Race

    NOT used for CSV generation.

    Args:
        logits: Raw model outputs.
        name: Model name for display.
    """
    model_name = "" if name is None else name
        
    gender_prob = torch.sigmoid(logits['gender']).item()
    gender_probs = [1 - gender_prob, gender_prob]  # 0 -> Male, 1 -> Female

    age_probs  = torch.softmax(logits['age'], dim=1)[0].cpu().tolist()
    race_probs = torch.softmax(logits['race'], dim=1)[0].cpu().tolist()
    
    # print("=" * 45)
    print(f"{"-"*5}{model_name} Prediction Result{"-"*5}")
    # print("=" * 45)

    predictions = {
        "Gender": print_ranked("Gender", gender_class, gender_probs),
        "Age":    print_ranked("Age", age_class, age_probs),
        "Race":   print_ranked("Race", race_class, race_probs),
    }

    print("\n\n")
    return predictions

In [7]:
def predict_image(model, processor, image_path, device):
    image = Image.open(image_path).convert("RGB")
    logits = get_logits(model, processor, image, device)
    return get_prediction_results(logits)


def predict_folder(models_dict, folder_path, device, output_csv,
                    extensions=(".jpg", ".jpeg", ".png","webp")):
    """
    models_dict: {
                "clip": (model, processor),
                "dinov2": (model, processor),
                "siglip": (model, processor)
            }
    """
    image_files = sorted(
        f for f in os.listdir(folder_path)
        if f.lower().endswith(extensions)
    )

    fieldnames = ["image", "model", "gender", "gender_prob",
                  "age", "age_prob", "race", "race_prob"]

    with open(output_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for image_file in image_files:
            image_path = os.path.join(folder_path, image_file)

            for model_name, (model, processor) in models_dict.items():
                result = predict_image(model, processor, image_path, device)
                writer.writerow({
                    "image": image_file,
                    "model": model_name,
                    **result,
                    
                })

    print(f"Saved predictions to {output_csv}")

## Load The Models

In [8]:
device = "cuda" if torch.cuda.is_available() else "cpu"

fairface_clipv2_model = load_checkpoint(
    FairFaceViT(clip_model),
    "clip",
    "v2",
    device
)

fairface_siglip_model = load_checkpoint(
    FairFaceViT(siglip_model),
    "siglip",
    "v2",
    device
)

fairface_dinov2_model = load_checkpoint(
    FairFaceViT(dinov2_model),
    "dinov2",
    "v2",
    device
)

# fairface_clipv2_1_model = load_checkpoint(
#     FairFaceViT(clip_model),
#     "clip",
#     "v2-1",
#     device
# )

In [9]:
models_dict = {
    "clip":    (fairface_clipv2_model, clip_processor),
    "dinov2":  (fairface_dinov2_model, dinov2_processor),
    "siglip":  (fairface_siglip_model, siglip_processor),
    # "clip-v2-1":    (fairface_clipv2_1_model, clip_processor),
}

In [10]:
# print(fairface_clip_model)

## Folder Inference

In [11]:
predict_folder(models_dict, "../test-images/", device, "predictions.csv")

Saved predictions to predictions.csv


## Single Image Inference

In [20]:
image = Image.open("../test-images/sepeher.jpg").convert("RGB")



In [ ]:
plt.figure(figsize=(3,2))
plt.imshow(image)
plt.axis('off')
plt.show()

device = "cuda" if torch.cuda.is_available() else "cpu"


clip_logits1 = get_logits(fairface_clipv2_model, clip_processor, image, device)
display_prediction(clip_logits1,'CLIP v2')

siglip_logits = get_logits(fairface_siglip_model, siglip_processor, image, device)
display_prediction(siglip_logits,'SIGLIP v2')

dinov2_logits = get_logits(fairface_dinov2_model, dinov2_processor, image, device)
display_prediction(dinov2_logits,'DINOV2 v2')


# clip_logits2 = get_logits(fairface_clipv2_1_model, clip_processor, image, device)
# display_prediction(clip_logits2,'CLIP v2-1')